In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import re

## 1. RNN-Based Text Generation

In [ ]:
# Dataset:
import requests

url = "https://www.gutenberg.org/files/11/11-0.txt"
raw_text = requests.get(url).text
raw_text = re.sub(r'\s+', ' ', raw_text)  # Replace multiple whitespace with a single space
raw_text = raw_text.strip()  # Remove leading and trailing whitespace
raw_text = raw_text[:10000]  # Use only the first 10,000 characters for this example
print(len(raw_text), raw_text[:500])

def tokenize(text):
    tokens = re.findall(r"\w+|[^\w\s*]", text)
    return tokens


processed_text = tokenize(raw_text)
print(f"Number of tokens: {len(processed_text)}, First 500 characters: {processed_text[:500]}")

def encoder(tokens):
    vocab = sorted(set(tokens))
    token_to_id = {token: idx for idx , token in enumerate(vocab)}
    encoded = [token_to_id[token] for token in tokens]
    return encoded, token_to_id


encoded_text, token_to_id = encoder(processed_text)
print(f"Vocabulary size: {len(token_to_id)}, First 100 tokens: {list(token_to_id.keys())[:100]} , First 100 encoded tokens: {encoded_text[:100]}")

# Dataset and DataLoader:
class TextDataset(Dataset):
    def __init__(self, text, seq_length):
        self.text = text
        self.seq_length = seq_length
        self.vocab = sorted(set(text))
        self.char_to_idx = {char: idx for idx, char in enumerate(self.vocab)}
        self.idx_to_char = {idx: char for idx, char in enumerate(self.vocab)}
    
    def __len__(self):
        return len(self.text) - self.seq_length
    
    def __getitem__(self, idx):
        input_seq = self.text[idx:idx+self.seq_length]
        target_seq = self.text[idx+1:idx+self.seq_length+1]
        input_indices = [self.char_to_idx[char] for char in input_seq]
        target_indices = [self.char_to_idx[char] for char in target_seq]
        return torch.tensor(input_indices), torch.tensor(target_indices)




10000 *** START OF THE PROJECT GUTENBERG EBOOK 11 *** [Illustration] Alice’s Adventures in Wonderland by Lewis Carroll THE MILLENNIUM FULCRUM EDITION 3.0 Contents CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster
Number of tokens: 2333, First 500 characters: ['START', 'OF', 'THE', 'PROJECT', 'GUTENBERG', 'EBOOK', '11', '[', 'Illustration', ']', 'Alice', '’', 's', 'Adventures', 'in', 'Wonderland', 'by', 'Lewis', 'Carroll', 'THE', 'MILLENNIUM', 'FULCRUM', 'EDITION', '3', '.', '0', 'Contents', 'CHAPTER', 'I', '.', 'Down', 'the', 'Rabbit', '-', 'Hole', 'CHAPTER', 'II', '.', 'The', 'Pool', 'of', 'Tears', 'CHAPTER', 'III', '.', 'A', 'Caucus', '-', 'Race', 'and', 'a', 'Long', 'Tale', 'CHAPTER', 'IV', '.'

This: 144696 *** start of the project gutenberg ebook 11 ***

[illustration]




alice’s adventures in wonderland

by lewis carroll

the millennium fulcrum edition 3.0

contents

 chapter i.     down the rabbit-hole
 chapter ii.    the pool of tears
 chapter iii.   a caucus-race and a long tale
 chapter iv.    the rabbit sends in a little bill
 chapter v.     advice from a caterpillar
 chapter vi.    pig and pepper
 chapter vii.   a mad tea-party
 chapter viii.  the queen’s croquet-ground
 chapter ix.    the


In [ ]:
# Model definition:

class TextGenerationModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(TextGenerationModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)  # *2 because of bidirectional
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        output = self.fc(output)
        return output